In [1]:
!pip install torch torchvision torchaudio

In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

import pandas as pd
from tqdm import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DATA_ROOT = Path.home() / "work"
TRAIN_DIR = DATA_ROOT / "train_val_data"
TEST_DIR = DATA_ROOT / "test_data"

print("Train dir exists:", TRAIN_DIR.exists())
print("Test dir exists:", TEST_DIR.exists())

PyTorch version: 2.10.0+cu128
CUDA available: False
Train dir exists: True
Test dir exists: True


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [4]:
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

Classes: ['Bicycle', 'Bridge', 'Bus', 'Car', 'Chimney', 'Crosswalk', 'Hydrant', 'Motorcycle', 'Other', 'Palm', 'Stair', 'Traffic Light']
Number of classes: 12


In [5]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 2400
Val samples: 600


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

import torch.nn as nn
import torch

class_weights = torch.tensor(
    [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0, 1.0, 1.0, 1.0, 1.0],
    device=device 
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Model ready")

Using device: cpu
Model ready


/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f}")

Epoch 1/5: 100%|██████████| 75/75 [02:40<00:00,  2.14s/it]


Epoch 1 - Train Loss: 1.2276


Epoch 2/5: 100%|██████████| 75/75 [02:43<00:00,  2.18s/it]


Epoch 2 - Train Loss: 0.4496


Epoch 3/5: 100%|██████████| 75/75 [02:41<00:00,  2.15s/it]


Epoch 3 - Train Loss: 0.2100


Epoch 4/5: 100%|██████████| 75/75 [02:43<00:00,  2.19s/it]


Epoch 4 - Train Loss: 0.1227


Epoch 5/5: 100%|██████████| 75/75 [02:41<00:00,  2.15s/it]

Epoch 5 - Train Loss: 0.0707


In [8]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = correct / total
print(f"Validation Accuracy: {val_acc:.4f}")

Validation Accuracy: 0.8183


In [9]:
test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  
    num_workers=2
)

print("Test samples:", len(test_dataset))

Test samples: 8730


In [10]:
model.eval()

all_probs = []
all_filenames = []

with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Predicting test data"):
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu())

Predicting test data: 100%|██████████| 273/273 [02:45<00:00,  1.65it/s]


In [11]:
all_probs = torch.cat(all_probs, dim=0).numpy()
print(all_probs.shape)

(8730, 12)


In [12]:
all_filenames = [Path(p).name for p, _ in test_dataset.samples]
print(all_filenames[:5])  
print(all_filenames[-5:])    


['03000.png', '03001.png', '03002.png', '03003.png', '03004.png']
['11726.png', '11727.png', '11728.png', '11729.png', '11730.png']


In [13]:
df = pd.DataFrame(
    all_probs,
    columns=class_names
)

df.insert(0, "ImageName", all_filenames)

df.head()

,ImageName,Bicycle,Bridge,Bus,Car,Chimney,Crosswalk,Hydrant,Motorcycle,Other,Palm,Stair,Traffic Light
0,03000.png,0.001443,0.001607,0.003065,0.969603,0.000519,0.004850,0.001059,0.002076,0.000786,0.014049,0.000595,0.000349
1,03001.png,0.102762,0.053918,0.018930,0.382262,0.008210,0.225136,0.003354,0.014702,0.061109,0.047924,0.019533,0.062160
2,03002.png,0.000319,0.000320,0.001166,0.995693,0.000248,0.000437,0.000256,0.000310,0.000151,0.000744,0.000154,0.000201
3,03003.png,0.000794,0.004001,0.972953,0.005356,0.000596,0.010411,0.000286,0.000242,0.000326,0.003582,0.000365,0.001088
4,03004.png,0.019219,0.011403,0.007139,0.544120,0.001564,0.033930,0.001959,0.036655,0.321829,0.001195,0.008261,0.012727


In [14]:
csv_path = Path.home() / "work" / "result.csv"
df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)

Saved to: /home/jovyan/work/result.csv


In [15]:
len(all_filenames)

8730

In [16]:
all_probs.shape


(8730, 12)

In [17]:
all_probs[0].sum()


np.float32(0.99999994)

In [19]:
import pandas as pd

df = pd.read_csv("result.csv")
print(df.shape)
print(df.head())


(8730, 13)
   ImageName   Bicycle    Bridge       Bus       Car   Chimney  Crosswalk  \
0  03000.png  0.001443  0.001607  0.003065  0.969603  0.000519   0.004850   
1  03001.png  0.102762  0.053918  0.018930  0.382262  0.008210   0.225136   
2  03002.png  0.000319  0.000320  0.001166  0.995693  0.000248   0.000437   
3  03003.png  0.000794  0.004001  0.972953  0.005356  0.000596   0.010411   
4  03004.png  0.019219  0.011403  0.007139  0.544120  0.001564   0.033930   

    Hydrant  Motorcycle     Other      Palm     Stair  Traffic Light  
0  0.001059    0.002076  0.000786  0.014049  0.000595       0.000349  
1  0.003354    0.014702  0.061109  0.047924  0.019533       0.062160  
2  0.000256    0.000310  0.000151  0.000744  0.000154       0.000201  
3  0.000286    0.000242  0.000326  0.003582  0.000365       0.001088  
4  0.001959    0.036655  0.321829  0.001195  0.008261       0.012727  
